In [57]:
import pyarrow as pa
import pandas as pd
import numpy as np
import zipfile
from pathlib import Path

In [58]:
RAW = Path("/Users/emrekaya/PycharmProjects/eu_pesticide_mrl_compliance/data/raw")
PROCESSED = Path("/Users/emrekaya/PycharmProjects/eu_pesticide_mrl_compliance/data/processed")

In [59]:
# there is 215 columns and i choose important ones for my business questions.
COLS = [
    "sampId_A",
    "sampCountry",
    "origCountry",
    "sampY", "sampM", "sampD",
    "sampMatCode.base.building",
    "paramCode.base.param",
    "resVal",
    "resLOQ",
    "resType",
    "evalCode",
]

In [60]:
# Combine for parquet files into one table
files = sorted(PROCESSED.glob("MOPER_*.parquet"))

parts = []
for file in files:
    parts.append(pd.read_parquet(file))

df = pd.concat(parts, ignore_index=True)
print("Total rows:", len(df))

df.to_parquet(PROCESSED / "samples_raw.parquet", index=False)

Total rows: 32387997


In [61]:
# Which years do we have for each country?
df.groupby("sampCountry")["sampY"].value_counts()

sampCountry  sampY
DE           2023     8530665
             2022     8026370
             2024     6265689
ES           2024      430587
             2023      381699
             2022      292340
             2021      273292
             2020      206179
FR           2022     3554078
             2023     2262188
             2024     2164910
Name: count, dtype: int64[pyarrow]

In [62]:
#One sample is tested for many pesticides, so rows != samples
n_rows = len(df)
n_samples = df["sampId_A"].nunique()

In [63]:
print("Rows:", n_rows)
print("Samples:", n_samples)
print("Analyses per sample:", round(n_rows / n_samples, 1))

Rows: 32387997
Samples: 103820
Analyses per sample: 312.0


In [64]:
#Result types and evaluation codes
print(df["resType"].value_counts())
print(df["evalCode"].value_counts())

resType
LOQ    18929867
LOD    13291981
VAL      140321
BIN       25828
Name: count, dtype: int64[pyarrow]
evalCode
J002A    27594947
J029A     4787522
J003A        3155
J031A        2373
Name: count, dtype: int64[pyarrow]


In [65]:
#Exceedance rate
# Drop rows that were never evaluated
ev = df[df["evalCode"] != "J029A"].copy()

In [66]:
# J003A = above MRL, J031A = above MRL but within measurement uncertainty
ev["exceeds"] = ev["evalCode"].isin(["J003A", "J031A"])

ev["non_compliant"] = ev["evalCode"] == "J003A"

In [67]:
# Roll up to sample level: a sample counts as exceeding if any of its
# analyses exceeded
samples = ev.groupby("sampId_A").agg(
    country=("sampCountry", "first"),
    year=("sampY", "first"),
    month=("sampM", "first"),
    origin=("origCountry", "first"),
    exceeds=("exceeds", "max"),
    non_compliant=("non_compliant", "max"),
)

In [68]:
print("Samples evaluated:", len(samples))
print("Exceedance rate %:", round(samples["exceeds"].mean() * 100, 2))
print("Non-compliance rate %:", round(samples["non_compliant"].mean() * 100, 2))

Samples evaluated: 97584
Exceedance rate %: 3.6
Non-compliance rate %: 2.0


In [69]:
###

In [70]:
# Are unevaluated results concentrated in one country?
df["not_evaluated"] = df["evalCode"] == "J029A"

In [71]:
print(df.groupby("sampCountry")["not_evaluated"].mean() * 100)

sampCountry
DE    13.077707
ES     0.479264
FR     22.49344
Name: not_evaluated, dtype: double[pyarrow]


In [72]:
#

In [73]:
# Exceedance rate by country and year
by_country = samples.groupby(["country", "year"])[["exceeds", "non_compliant"]].mean() * 100
by_country["n"] = samples.groupby(["country", "year"]).size()

In [74]:
print(by_country.round(2))

              exceeds  non_compliant      n
country year                               
DE      2022     4.64           2.45  19139
        2023     3.44            1.8  19601
        2024     2.35           1.24  16075
ES      2020     3.82           1.75   1543
        2021     3.99           2.52   1905
        2022     1.66           0.75   1743
        2023     2.94           1.43   2998
        2024     2.32           1.13   3626
FR      2022     3.71           2.43  13811
        2023     3.49           2.17  10552
        2024     5.37           3.03   6591


In [75]:
#

In [76]:
# Which origins have the highest exceedance rate?
by_origin = samples.groupby("origin")["exceeds"].mean() * 100
counts = samples.groupby("origin").size()

In [77]:
result = pd.DataFrame({"exceedance_%": by_origin.round(2), "n": counts})
result = result[result["n"] >= 200]

In [78]:
print(result.sort_values("exceedance_%", ascending=False).head(20))

        exceedance_%     n
origin                    
LK             28.62   318
VN             21.72   732
PK             21.49   228
TH             13.71   248
DO             13.30   203
IN             12.06  2371
EG             10.25  1200
TR              9.92  1814
EC              9.31   247
KE              9.18  1590
CN              8.69  1727
BR              8.33   840
GP              7.13   996
CO              6.05   446
MA              5.72  1417
US              5.32   263
PE              5.15   816
MQ              4.98   723
XX              4.83  8201
AR              4.56   241


In [79]:
#

In [80]:
# Is there a seasonal pattern?
by_month = samples.groupby("month")["exceeds"].mean() * 100

In [81]:
print(by_month.round(2))

month
1     4.71
2     4.27
3     4.14
4     3.39
5     3.32
6     3.25
7     3.44
8     3.48
9     3.35
10    3.29
11    3.26
12    4.08
Name: exceeds, dtype: float64


In [82]:
#

In [83]:
# Is the winter peak driven by imports?
samples["is_eu"] = samples["origin"].isin(["ES","FR","DE","IT","NL","PT","BE","PL","GR"])

In [84]:
print(samples.groupby(["month", "is_eu"])["exceeds"].mean().unstack() * 100)

is_eu     False     True 
month                    
1      8.639483  1.852941
2      8.384195  1.605839
3      8.181818  1.611952
4      6.751502  1.685192
5      7.423241  1.607143
6      7.821901  1.579871
7      8.170311  1.555115
8      9.351504  1.251788
9      8.019246  1.724857
10     7.300509  1.602289
11     6.468401  1.857283
12     7.740586  1.792742


In [85]:
# Does the share of non-EU samples change by month?
share = samples.groupby("month")["is_eu"].mean()

In [86]:
print((1 - share).round(3) * 100)   # non-EU share, %

month
1     42.1
2     39.4
3     38.5
4     33.7
5     29.5
6     26.7
7     28.4
8     27.6
9     25.8
10    29.6
11    30.5
12    38.5
Name: is_eu, dtype: float64
